# 04 · 5-minute nowcast

**Purpose.** Short-term power for the dashboard and for battery/load decisions: the mean power over the next 5 minutes.

```
 last 15 min of cleaned 1-min means (power, wind, irradiance) + known future clear-sky ─► LightGBM ─► next 5 min
```
Baselines: persistence (last minute) and, for PV, *smart persistence* — the last 15 min clear-sky index times the
clear-sky irradiance 5 min ahead, the standard solar nowcast reference.

In [1]:
%matplotlib inline
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from aerosolar.config import Config, REPO_ROOT
from aerosolar.sim.realism import _style, S1, S2, S3, INK, INK2
_style(); plt.rcParams['figure.dpi'] = 110
cfg = Config.load()
DATA = REPO_ROOT / 'data'
pd.set_option('display.width', 140); pd.set_option('display.max_columns', 30)
from aerosolar.ml.train import load_latest, load_split
bundle, meta = load_latest()
print('model version', meta['version'])

model version 20261001T125452Z


In [2]:
import json
m = json.loads((REPO_ROOT / 'reports' / 'forecast_metrics.json').read_text())['nowcast']
pd.DataFrame(m).T[['mae_w', 'r2', 'cap_accuracy_rmse', 'skill_vs_persistence', 'skill_vs_smart_persistence']].round(3)

,mae_w,r2,cap_accuracy_rmse,skill_vs_persistence,skill_vs_smart_persistence
pv,0.027,0.999,0.992,0.579,0.855
wind,0.286,0.690,0.951,0.344,NaN
total,0.303,0.989,0.980,0.369,0.467


In [3]:
_, nc = load_split('test')
d = nc[(nc['ts'] >= pd.Timestamp('2026-04-15', tz='UTC')) & (nc['ts'] < pd.Timestamp('2026-04-16', tz='UTC'))].dropna(subset=['y_total'])
p = bundle['nowcaster'].predict(d)
x = d['ts'].dt.tz_convert(None) + pd.Timedelta(hours=4)
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(x, d['y_total'], color=INK, lw=0.8, label='actual (next 5 min mean)'); ax.plot(x, p['nowcast_total'], color=S1, lw=0.8, label='nowcast')
ax.set_ylabel('total power [W]'); ax.legend(fontsize=8); plt.show()

**What to notice.** The nowcast tracks the day closely (R² ≈ 0.99 for total power). Its skill over persistence is
modest because 5 minutes is short: most of the value is in getting the sunrise/sunset ramps and gust periods right.

## References
* Romano, Patterson, Candès (2019). *Conformalized quantile regression.* NeurIPS.
* Ke et al. (2017). *LightGBM: a highly efficient gradient boosting decision tree.* NeurIPS.
* Yang et al. (2020). *Verification of deterministic solar forecasts.* Solar Energy 210 (clear-sky index, smart persistence, skill scores).
* Hong et al. (2016). *Probabilistic energy forecasting: GEFCom2014 and beyond.* IJF 32.
* Haurwitz (1945). *Insolation in relation to cloudiness and cloud density.* J. Meteorology.
* Kamoji, Kedare, Prabhu (2009). *Performance tests on helical Savonius rotors.* Renewable Energy 34.